In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langgraph.graph import StateGraph,START,END
from langchain_openai import ChatOpenAI
from typing import TypedDict

In [3]:
# to implement the memeory
from langgraph.checkpoint.memory import InMemorySaver

In [4]:
class jokeState(TypedDict):
    topic: str
    joke: str
    explanation: str

In [5]:
graph = StateGraph(state_schema=jokeState)

In [6]:
llm = ChatOpenAI()

In [7]:
def generate_joke(state:jokeState):
    topic = state['topic']
    prompt = f"generate a joke on the following topic {topic}"
    joke = llm.invoke(prompt).content
    return {'joke':joke}

In [8]:
def explain_joke(state:jokeState):
    topic = state['topic']
    joke = state['joke']

    prompt = f'give the explanation of the following joke \n{joke} which is on the topic {topic}'
    explanation = llm.invoke(prompt).content

    return {'explanation':explanation}

In [9]:
graph.add_node('generate_joke',generate_joke)
graph.add_node('explain_joke',explain_joke)

graph.add_edge(START,'generate_joke')
graph.add_edge('generate_joke','explain_joke')
graph.add_edge('explain_joke',END)


In [10]:
checkpointer = InMemorySaver()
workflow = graph.compile(checkpointer=checkpointer)

In [12]:
workflow.invoke(input={
    'topic':'india'
},config={'configurable':{'thread_id':1}})

{'topic': 'india',
 'joke': 'Why did the Indian man bring a ladder to the bar? \nHe heard they had a high spirits selection!',
 'explanation': 'This joke plays on the double meaning of the word "high." In this context, "high spirits" refers to a wide variety of alcoholic beverages that are available at the bar. However, the Indian man interpreted "high spirits" literally, thinking that the bar was physically located at a high elevation. So, he brought a ladder to help him reach the high bar. This joke draws on the stereotype of Indian people being resourceful and clever in finding solutions to everyday problems.'}

In [22]:
workflow.get_state(config={'configurable':{'thread_id':1}})

StateSnapshot(values={'topic': 'india', 'joke': 'Why did the Indian man bring a ladder to the bar? \nHe heard they had a high spirits selection!', 'explanation': 'This joke plays on the double meaning of the word "high." In this context, "high spirits" refers to a wide variety of alcoholic beverages that are available at the bar. However, the Indian man interpreted "high spirits" literally, thinking that the bar was physically located at a high elevation. So, he brought a ladder to help him reach the high bar. This joke draws on the stereotype of Indian people being resourceful and clever in finding solutions to everyday problems.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f181b49-a3f9-6da9-8002-322dfc0c98c8'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-07-17T07:53:30.715267+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f181b49-9783-6271-8001-ff02742f914

In [23]:
list(workflow.get_state_history(config={'configurable':{'thread_id':1}}))

[StateSnapshot(values={'topic': 'india', 'joke': 'Why did the Indian man bring a ladder to the bar? \nHe heard they had a high spirits selection!', 'explanation': 'This joke plays on the double meaning of the word "high." In this context, "high spirits" refers to a wide variety of alcoholic beverages that are available at the bar. However, the Indian man interpreted "high spirits" literally, thinking that the bar was physically located at a high elevation. So, he brought a ladder to help him reach the high bar. This joke draws on the stereotype of Indian people being resourceful and clever in finding solutions to everyday problems.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f181b49-a3f9-6da9-8002-322dfc0c98c8'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-07-17T07:53:30.715267+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f181b49-9783-6271-8001-ff02742f91

In [24]:
workflow.invoke(input={
    'topic':'pizza'
},config={'configurable':{'thread_id':2}})

{'topic': 'pizza',
 'joke': 'Why did the pizza go to the therapist?\n\nBecause it felt like it was topping out on life.',
 'explanation': 'This joke is a play on words. Typically, when someone is "topping out on life," it means they feel like they have reached their peak or are not able to improve any further. In the case of the pizza, "topping out" has a double meaning because pizzas are often topped with various ingredients. So the pizza went to the therapist because it felt like it was topping out on life, both in terms of feeling like it couldn\'t improve anymore and in terms of having too many toppings.'}

In [25]:
workflow.get_state(config={'configurable':{'thread_id':2}})

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the therapist?\n\nBecause it felt like it was topping out on life.', 'explanation': 'This joke is a play on words. Typically, when someone is "topping out on life," it means they feel like they have reached their peak or are not able to improve any further. In the case of the pizza, "topping out" has a double meaning because pizzas are often topped with various ingredients. So the pizza went to the therapist because it felt like it was topping out on life, both in terms of feeling like it couldn\'t improve anymore and in terms of having too many toppings.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f181b4c-dd23-6047-8006-62a9507bd312'}}, metadata={'source': 'loop', 'step': 6, 'parents': {}}, created_at='2026-07-17T07:54:57.239644+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f181b4c-d15d-6494-8005-5476798a381c'}}

In [26]:
list(workflow.get_state_history(config = {'configurable':{'thread_id':2}}))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the therapist?\n\nBecause it felt like it was topping out on life.', 'explanation': 'This joke is a play on words. Typically, when someone is "topping out on life," it means they feel like they have reached their peak or are not able to improve any further. In the case of the pizza, "topping out" has a double meaning because pizzas are often topped with various ingredients. So the pizza went to the therapist because it felt like it was topping out on life, both in terms of feeling like it couldn\'t improve anymore and in terms of having too many toppings.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f181b4c-dd23-6047-8006-62a9507bd312'}}, metadata={'source': 'loop', 'step': 6, 'parents': {}}, created_at='2026-07-17T07:54:57.239644+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f181b4c-d15d-6494-8005-5476798a381c'}

## TimeTravel

In [27]:
workflow.get_state(config={'configurable':{'thread_id':2,'checkpoint_id':'1f181b49-acd3-69ef-8001-a55be6221906'}})

StateSnapshot(values={'topic': 'pizza', 'joke': "Why did the pizza maker go to therapy? Because he couldn't stop topping himself!"}, next=('explain_joke',), config={'configurable': {'thread_id': '2', 'checkpoint_id': '1f181b49-acd3-69ef-8001-a55be6221906'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-07-17T07:53:31.643280+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f181b49-a498-664a-8000-ed7d8b2ca9b2'}}, tasks=(PregelTask(id='cea29134-f476-9026-c38b-2c639655daae', name='explain_joke', path=('__pregel_pull', 'explain_joke'), error=None, interrupts=(), state=None, result={'explanation': 'This joke plays on the double meaning of "topping." In the context of making pizza, "topping" refers to adding ingredients to the pizza, such as cheese, vegetables, and meat. However, "topping yourself" also means committing suicide. So, the joke is suggesting that the pizza maker went to therapy because he couldn\'t stop 

In [28]:
workflow.invoke(input=None,config={'configurable':{'thread_id':2,'checkpoint_id':'1f181b49-acd3-69ef-8001-a55be6221906'}})

{'topic': 'pizza',
 'joke': "Why did the pizza maker go to therapy? Because he couldn't stop topping himself!",
 'explanation': 'This joke plays on the double meaning of the phrase "topping yourself." In the context of making pizza, "topping yourself" means adding toppings to a pizza. However, in a more serious context, "topping yourself" can mean committing suicide. The humor comes from the pun and the absurdity of a pizza maker seeking therapy because he can\'t stop adding toppings to his pizzas.'}

In [29]:
list(workflow.get_state_history(config = {'configurable':{'thread_id':2}}))

[StateSnapshot(values={'topic': 'pizza', 'joke': "Why did the pizza maker go to therapy? Because he couldn't stop topping himself!", 'explanation': 'This joke plays on the double meaning of the phrase "topping yourself." In the context of making pizza, "topping yourself" means adding toppings to a pizza. However, in a more serious context, "topping yourself" can mean committing suicide. The humor comes from the pun and the absurdity of a pizza maker seeking therapy because he can\'t stop adding toppings to his pizzas.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f181b52-8e57-61e4-8003-855934569f24'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-07-17T07:57:30.038508+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f181b52-742c-642d-8002-01708efc46ce'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': "Why did the pizza maker go to the

## Updating State

In [35]:
workflow.update_state(config={'configurable':{'thread_id':2,'checkpoint_id':'1f181b49-a498-664a-8000-ed7d8b2ca9b2','checkpoint_ns':''}},values={'topic':'burger'},)

{'configurable': {'thread_id': '2',
  'checkpoint_ns': '',
  'checkpoint_id': '1f181b5c-8c74-66b9-8001-e4dff34983a0'}}

In [38]:
list(workflow.get_state_history(config={'configurable':{'thread_id':2}}))

[StateSnapshot(values={'topic': 'burger'}, next=('generate_joke',), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f181b5c-8c74-66b9-8001-e4dff34983a0'}}, metadata={'source': 'update', 'step': 1, 'parents': {}}, created_at='2026-07-17T08:01:58.276237+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f181b49-a498-664a-8000-ed7d8b2ca9b2'}}, tasks=(PregelTask(id='97d30c5c-b852-e218-54f4-3f5690e35e44', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'burger'}, next=('generate_joke',), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f181b5c-1b5f-6a50-8001-71cd6dcae4e2'}}, metadata={'source': 'update', 'step': 1, 'parents': {}}, created_at='2026-07-17T08:01:46.418820+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id

In [39]:
workflow.invoke(input=None,config ={'configurable': {'thread_id': '2',
  'checkpoint_ns': '',
  'checkpoint_id': '1f181b5c-8c74-66b9-8001-e4dff34983a0'}})

{'topic': 'burger',
 'joke': "Why did the burger go to the party alone? It couldn't find a good bun to go with it!",
 'explanation': "This joke plays on the idea that burgers are typically served between two buns. The punchline humorously suggests that the burger couldn't find a good bun to accompany it to the party, implying that without a bun, the burger felt incomplete or out of place."}

In [40]:
list(workflow.get_state_history(config = {'configurable':{'thread_id':2}}))

[StateSnapshot(values={'topic': 'burger', 'joke': "Why did the burger go to the party alone? It couldn't find a good bun to go with it!", 'explanation': "This joke plays on the idea that burgers are typically served between two buns. The punchline humorously suggests that the burger couldn't find a good bun to accompany it to the party, implying that without a bun, the burger felt incomplete or out of place."}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f181b64-0e4a-6539-8003-27b1d6506fa5'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-07-17T08:05:19.795314+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f181b64-036d-66dd-8002-e644b52707b6'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'burger', 'joke': "Why did the burger go to the party alone? It couldn't find a good bun to go with it!"}, next=('explain_joke',), config={'configurable': {'thread

## Fault Tolerance

In [41]:

from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time

In [42]:
    # 1. Define the state
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str

In [43]:
# 2. Define steps
def step_1(state: CrashState) -> CrashState:
    print("✅ Step 1 executed")
    return {"step1": "done", "input": state["input"]}

def step_2(state: CrashState) -> CrashState:
    print("⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)")
    time.sleep(1000)  # Simulate long-running hang
    return {"step2": "done"}

def step_3(state: CrashState) -> CrashState:
    print("✅ Step 3 executed")
    return {"done": True}

In [44]:

# 3. Build the graph
builder = StateGraph(CrashState)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.set_entry_point("step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [ ]:
try:
    print("▶️ Running graph: Please manually interrupt during Step 2...")
    graph.invoke({"input": "start"}, config={"configurable": {"thread_id": 'thread-1'}})
except KeyboardInterrupt:
    print("❌ Kernel manually interrupted (crash simulated).")

In [ ]:
# 6. Re-run to show fault-tolerant resume
print("\n🔁 Re-running the graph to demonstrate fault tolerance...")
final_state = graph.invoke(None, config={"configurable": {"thread_id": 'thread-1'}})
print("\n✅ Final State:", final_state)

In [ ]:

list(graph.get_state_history({"configurable": {"thread_id": 'thread-1'}}))